# Catálogo de datos: descripciones en Unity Catalog
Agrega un comentario a cada tabla y a cada columna de `dunnhumby.bronze`, `dunnhumby.silver` y `dunnhumby.gold`. Así el Catalog Explorer muestra, para cada tabla, qué contiene y qué significa cada columna.

**Va al final del Workflow.** Silver y Gold se actualizan con `MERGE`, que conserva los comentarios, pero en la primera corrida las tablas se crean sin ellos y Bronze se reescribe completo en cada carga. Al correr este notebook después de Gold, los comentarios siempre quedan al día. Volver a ejecutarlo no cambia nada.


In [0]:
# SECCIÓN 0 · Funciones comunes del catálogo
def sql_texto(s):
    """Escapa las comillas simples para usar el texto dentro de un literal SQL."""
    return s.replace("\\", "\\\\").replace("'", "\\'")


def documentar_tabla(esquema, tabla, descripcion, columnas):
    """Pone el comentario de la tabla y de cada columna. Falla si una columna real queda sin describir."""
    nombre = f"dunnhumby.{esquema}.{tabla}"
    reales = spark.table(nombre).columns

    # Las dos listas deben coincidir: así el catálogo no se desactualiza si cambia el código de Silver o Gold
    sin_describir = [c for c in reales if c not in columnas]
    inexistentes = [c for c in columnas if c not in reales]
    assert not sin_describir, f"{nombre}: columnas sin descripción {sin_describir}"
    assert not inexistentes, f"{nombre}: columnas descritas que no existen {inexistentes}"

    spark.sql(f"COMMENT ON TABLE {nombre} IS '{sql_texto(descripcion)}'")
    for c in reales:
        spark.sql(f"ALTER TABLE {nombre} ALTER COLUMN `{c}` COMMENT '{sql_texto(columnas[c])}'")
    print(f"{nombre}: tabla y {len(reales)} columnas documentadas")


## 1. Bronze
En Bronze las columnas son el CSV original como texto; se documentan la tabla y las dos columnas de auditoría. El significado de cada columna se documenta en Silver, donde ya tiene su tipo.

In [0]:
# SECCIÓN 1 · Bronze · comentario de tabla y columnas de auditoría
BRONZE = {'campaign_desc': 'Crudo de campaign_desc.csv (Kaggle). Todas las columnas como texto. Una fila por campaña de marketing.',
 'campaign_table': 'Crudo de campaign_table.csv (Kaggle). Todas las columnas como texto. Qué hogares recibieron cada campaña.',
 'coupon': 'Crudo de coupon.csv (Kaggle). Todas las columnas como texto. Productos a los que aplica cada cupón por campaña.',
 'coupon_redempt': 'Crudo de coupon_redempt.csv (Kaggle). Todas las columnas como texto. Cupones canjeados por los hogares.',
 'hh_demographic': 'Crudo de hh_demographic.csv (Kaggle). Todas las columnas como texto. Datos demográficos de 801 hogares.',
 'product': 'Crudo de product.csv (Kaggle). Todas las columnas como texto. Catálogo de productos.',
 'transaction_data': 'Crudo de transaction_data.csv (Kaggle). Todas las columnas como texto. Una fila por producto comprado en '
                     'una canasta.'}
AUDITORIA_BRONZE = {'_archivo_origen': 'Auditoría: ruta del CSV del Volume del que vino la fila',
 '_fecha_carga': 'Auditoría: fecha y hora en que se cargó la fila en Bronze'}

for tabla, descripcion in BRONZE.items():
    nombre = f"dunnhumby.bronze.{tabla}"
    spark.sql(f"COMMENT ON TABLE {nombre} IS '{sql_texto(descripcion)}'")
    for c, d in AUDITORIA_BRONZE.items():
        spark.sql(f"ALTER TABLE {nombre} ALTER COLUMN `{c}` COMMENT '{sql_texto(d)}'")
    print(f"{nombre}: tabla y columnas de auditoría documentadas")


## 2. Silver
Tablas limpias y tipadas. Las reglas citadas (regla 4, regla 8…) son las de `02_silver`.

In [0]:
# SECCIÓN 2 · Silver · tabla y todas sus columnas
SILVER = {'campaign_desc': ('Campañas de marketing con su tipo y vigencia. Limpieza: tipos y texto sin espacios.',
                   {'description': 'Tipo de campaña: TypeA, TypeB o TypeC',
                    'campaign': 'Identificador de la campaña de marketing (1 a 30)',
                    'start_day': 'Día de inicio de la campaña (misma escala que day)',
                    'end_day': 'Día de fin de la campaña (misma escala que day)',
                    '_fecha_proceso': 'Auditoría: fecha y hora en que la fila se insertó o cambió por última vez en esta capa'}),
 'campaign_table': ('Hogares que recibieron cada campaña. Una fila por hogar y campaña.',
                    {'description': 'Tipo de campaña: TypeA, TypeB o TypeC',
                     'household_key': 'Identificador único del hogar (cliente con tarjeta de fidelización)',
                     'campaign': 'Identificador de la campaña de marketing (1 a 30)',
                     '_fecha_proceso': 'Auditoría: fecha y hora en que la fila se insertó o cambió por última vez en esta capa'}),
 'coupon': ('Productos a los que aplica cada cupón en cada campaña. Sin filas duplicadas (regla 12).',
            {'coupon_upc': 'Código único del cupón',
             'product_id': 'Identificador único del producto',
             'campaign': 'Identificador de la campaña de marketing (1 a 30)',
             '_fecha_proceso': 'Auditoría: fecha y hora en que la fila se insertó o cambió por última vez en esta capa'}),
 'coupon_redempt': ('Canjes de cupones: qué hogar canjeó qué cupón, qué día y en qué campaña. Carga incremental por día con '
                    'MERGE.',
                    {'household_key': 'Identificador único del hogar (cliente con tarjeta de fidelización)',
                     'day': 'Día del canje (1 a 711)',
                     'coupon_upc': 'Código único del cupón',
                     'campaign': 'Identificador de la campaña de marketing (1 a 30)',
                     '_fecha_proceso': 'Auditoría: fecha y hora en que la fila se insertó o cambió por última vez en esta capa'}),
 'hh_demographic': ('Perfil demográfico de 801 hogares (32% del panel). Los valores de relleno pasan a NULL (regla 10).',
                    {'age_desc': 'Rango de edad estimado del jefe de hogar',
                     'marital_status_code': 'Estado civil: A o B según la guía del dataset; NULL si era U (desconocido)',
                     'income_desc': 'Rango de ingreso anual del hogar (por ejemplo 35-49K)',
                     'homeowner_desc': 'Tenencia de la vivienda; NULL si era Unknown',
                     'hh_comp_desc': 'Composición del hogar (adultos y niños); NULL si era Unknown',
                     'household_size_desc': 'Número de personas en el hogar (1 a 5+)',
                     'kid_category_desc': 'Número de niños; None/Unknown se conserva porque mezcla dos significados',
                     'household_key': 'Identificador único del hogar (cliente con tarjeta de fidelización)',
                     '_fecha_proceso': 'Auditoría: fecha y hora en que la fila se insertó o cambió por última vez en esta capa',
                     'income_orden': 'Orden numérico del rango de ingreso (1 = Under 15K, 12 = 250K+), para ordenar '
                                     'correctamente (regla 11)'}),
 'product': ('Catálogo de productos con su jerarquía. Productos sin clasificar etiquetados y marca de no mercadería.',
             {'product_id': 'Identificador único del producto',
              'manufacturer': 'Código del fabricante',
              'department': 'Departamento del producto; SIN CLASIFICAR si venía vacío (regla 8)',
              'brand': 'Tipo de marca: National (marca de fabricante) o Private (marca propia)',
              'commodity_desc': 'Categoría del producto; SIN CLASIFICAR si venía vacío',
              'sub_commodity_desc': 'Subcategoría del producto; SIN CLASIFICAR si venía vacío',
              'curr_size_of_product': 'Tamaño o presentación del producto; NULL si venía vacío (regla 7)',
              '_fecha_proceso': 'Auditoría: fecha y hora en que la fila se insertó o cambió por última vez en esta capa',
              'es_no_mercaderia': 'True para KIOSK-GAS, MISC SALES TRAN y MISC. TRANS., cuya cantidad no es comparable (regla '
                                  '9)'}),
 'transaction_data': ('Líneas de compra limpias: una fila por producto en una canasta, solo con cantidad mayor que 0. Carga '
                      'incremental por día con MERGE por basket_id + product_id. Tabla central del modelo.',
                      {'household_key': 'Identificador único del hogar (cliente con tarjeta de fidelización)',
                       'basket_id': 'Identificador único de la canasta (una visita de compra)',
                       'day': 'Día de la transacción, relativo al inicio del estudio (1 a 711)',
                       'product_id': 'Identificador único del producto',
                       'quantity': 'Unidades compradas del producto en la canasta',
                       'sales_value': 'Monto cobrado por la línea, en dólares; redondeado a 2 decimales en Silver',
                       'store_id': 'Identificador de la tienda',
                       'retail_disc': 'Descuento por tarjeta de fidelización; negativo cuando hay descuento',
                       'trans_time': 'Hora de la transacción en formato HHMM (45 = 00:45)',
                       'week_no': 'Semana de la transacción, relativa al inicio del estudio (1 a 102)',
                       'coupon_disc': 'Descuento por cupón; negativo cuando se aplicó',
                       'coupon_match_disc': 'Descuento de la tienda que iguala el cupón del fabricante; negativo cuando se '
                                            'aplicó',
                       '_fecha_proceso': 'Auditoría: fecha y hora en que la fila se insertó o cambió por última vez en esta capa',
                       'hora': 'Hora del día de la transacción (0 a 23), derivada de trans_time'}),
 'transaction_data_cuarentena': ('Líneas de transacción rechazadas por tener cantidad 0 (regla 4), con su motivo. Se conservan '
                                 'para auditoría. Carga incremental por día con MERGE.',
                                 {'household_key': 'Identificador único del hogar (cliente con tarjeta de fidelización)',
                                  'basket_id': 'Identificador único de la canasta (una visita de compra)',
                                  'day': 'Día de la transacción, relativo al inicio del estudio (1 a 711)',
                                  'product_id': 'Identificador único del producto',
                                  'quantity': 'Unidades compradas del producto en la canasta',
                                  'sales_value': 'Monto cobrado por la línea, en dólares; redondeado a 2 decimales en Silver',
                                  'store_id': 'Identificador de la tienda',
                                  'retail_disc': 'Descuento por tarjeta de fidelización; negativo cuando hay descuento',
                                  'trans_time': 'Hora de la transacción en formato HHMM (45 = 00:45)',
                                  'week_no': 'Semana de la transacción, relativa al inicio del estudio (1 a 102)',
                                  'coupon_disc': 'Descuento por cupón; negativo cuando se aplicó',
                                  'coupon_match_disc': 'Descuento de la tienda que iguala el cupón del fabricante; negativo '
                                                       'cuando se aplicó',
                                  '_fecha_proceso': 'Auditoría: fecha y hora en que la fila se insertó o cambió por última vez '
                                                    'en esta capa',
                                  'hora': 'Hora del día de la transacción (0 a 23), derivada de trans_time',
                                  'motivo': 'Motivo del rechazo: cantidad y venta en cero, o cantidad cero con venta positiva'})}

for tabla, (descripcion, columnas) in SILVER.items():
    documentar_tabla("silver", tabla, descripcion, columnas)


## 3. Gold
Una o más tablas por cada pregunta de negocio de `03_gold`.

In [0]:
# SECCIÓN 3 · Gold · tabla y todas sus columnas
GOLD = {'ventas_departamento_periodo': ('Pregunta 1: ventas por departamento y periodo de 4 semanas, sin no mercadería. Incremental: '
                                 'solo se recalculan los periodos con datos nuevos.',
                                 {'periodo': 'Periodo de 4 semanas (1 a 26); el 26 solo tiene 2 semanas',
                                  'department': 'Departamento del producto',
                                  'ventas': 'Suma de sales_value del departamento en el periodo',
                                  'lineas': 'Número de líneas de transacción',
                                  'canastas': 'Número de canastas distintas con productos del departamento',
                                  'semanas_en_periodo': 'Semanas con datos en el periodo; menos de 4 indica un periodo '
                                                        'incompleto',
                                  '_fecha_proceso': 'Auditoría: fecha y hora en que la fila se insertó o cambió por última vez '
                                                    'en esta capa'}),
 'gasto_hogar_cupones': ('Pregunta 2: una fila por hogar con su gasto, cupones canjeados y campañas recibidas.',
                         {'household_key': 'Identificador único del hogar (cliente con tarjeta de fidelización)',
                          'ventas_total': 'Gasto total del hogar en todo el periodo (suma de sales_value)',
                          'descuento_cupon_total': 'Descuento total por cupones (coupon_disc + coupon_match_disc), en positivo',
                          'n_canastas': 'Número de canastas (visitas) del hogar',
                          'n_cupones_canjeados': 'Número de cupones canjeados; 0 si no canjeó',
                          'n_campanas_recibidas': 'Número de campañas distintas que recibió; 0 si ninguna',
                          'canjeo_cupon': 'True si el hogar canjeó al menos un cupón',
                          '_fecha_proceso': 'Auditoría: fecha y hora en que la fila se insertó o cambió por última vez en esta '
                                            'capa'}),
 'gasto_por_perfil_hogar': ('Pregunta 3: gasto por hogar según ingreso y composición, incluido el grupo sin datos demográficos.',
                            {'dimension': 'Dimensión de análisis: ingreso o composicion_hogar',
                             'categoria': 'Valor de la dimensión; incluye Sin datos demográficos y Composición desconocida',
                             'orden': 'Orden de presentación de la categoría (99 = sin datos demográficos)',
                             'n_hogares': 'Número de hogares en la categoría',
                             'ventas_total': 'Gasto total de los hogares de la categoría',
                             'gasto_promedio_hogar': 'Gasto promedio por hogar',
                             'gasto_mediano_hogar': 'Gasto mediano por hogar (aproximado), menos sensible a hogares muy grandes',
                             'canastas_promedio': 'Canastas promedio por hogar',
                             'gasto_por_canasta': 'Gasto total dividido entre canastas (ticket promedio)',
                             '_fecha_proceso': 'Auditoría: fecha y hora en que la fila se insertó o cambió por última vez en '
                                               'esta capa'}),
 'segmentos_hogar': ('Pregunta 4: una fila por hogar con su decil de gasto y su estado de actividad.',
                     {'household_key': 'Identificador único del hogar (cliente con tarjeta de fidelización)',
                      'ventas_total': 'Gasto total del hogar en todo el periodo',
                      'n_canastas': 'Número de canastas del hogar',
                      'ultima_semana': 'Última semana con compra',
                      'semanas_sin_comprar': 'Semanas desde la última compra hasta la semana 102',
                      'estado_actividad': 'Activo (0 a 3 semanas sin comprar), En riesgo (4 a 12) o Inactivo (13 o más)',
                      'decil_gasto': 'Decil de gasto: 1 es el 10% de hogares que más gasta',
                      '_fecha_proceso': 'Auditoría: fecha y hora en que la fila se insertó o cambió por última vez en esta '
                                        'capa'})}

for tabla, (descripcion, columnas) in GOLD.items():
    documentar_tabla("gold", tabla, descripcion, columnas)


## 4. Verificación y catálogo resultante
Se lee lo que quedó guardado en Unity Catalog (`information_schema`), no lo que dice este notebook.

In [0]:
# SECCIÓN 4 · Verificación: ninguna tabla ni columna de las tres capas debe quedar sin comentario
# Esperado: 0 tablas y 0 columnas sin comentario en silver y gold; en bronze solo las columnas del CSV
sin_comentario_tablas = spark.sql("""
    SELECT table_schema, table_name FROM dunnhumby.information_schema.tables
    WHERE table_schema IN ('bronze', 'silver', 'gold') AND (comment IS NULL OR comment = '')
""")
sin_comentario_cols = spark.sql("""
    SELECT table_schema, table_name, column_name FROM dunnhumby.information_schema.columns
    WHERE table_schema IN ('silver', 'gold') AND (comment IS NULL OR comment = '')
""")
assert sin_comentario_tablas.count() == 0, "Hay tablas sin comentario"
assert sin_comentario_cols.count() == 0, "Hay columnas de silver o gold sin comentario"
print("Todas las tablas y columnas de silver y gold tienen descripción")


In [0]:
# SECCIÓN 4 · Catálogo de tablas (evidencia para la presentación)
display(spark.sql("""
    SELECT table_schema AS capa, table_name AS tabla, comment AS descripcion
    FROM dunnhumby.information_schema.tables
    WHERE table_schema IN ('bronze', 'silver', 'gold')
    ORDER BY CASE table_schema WHEN 'bronze' THEN 1 WHEN 'silver' THEN 2 ELSE 3 END, table_name
"""))


In [0]:
# SECCIÓN 4 · Diccionario de columnas de silver y gold, con el tipo real guardado en Delta
display(spark.sql("""
    SELECT table_schema AS capa, table_name AS tabla, column_name AS columna,
           full_data_type AS tipo, comment AS descripcion
    FROM dunnhumby.information_schema.columns
    WHERE table_schema IN ('silver', 'gold')
    ORDER BY table_schema DESC, table_name, ordinal_position
"""))
